In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)

d = np.load("../data/processed/windows_fd001.npz")
RUL_CAP = 125.0

def to_tensors(X, y):
    # targets are divided by 125 so the model learns values between 0 and 1
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y / RUL_CAP, dtype=torch.float32)

X_tr, y_tr = to_tensors(d["X_tr"], d["y_tr"])
X_val, y_val = to_tensors(d["X_val"], d["y_val"])
X_test, y_test = to_tensors(d["X_test"], d["y_test"])

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=128, shuffle=True)


class LSTMRegressor(nn.Module):
    def __init__(self, n_features=28, hidden=64, layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, num_layers=layers,
                            batch_first=True, dropout=dropout)
        self.head = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, x):
        out, _ = self.lstm(x)           # out: (batch, 30, hidden)
        return self.head(out[:, -1, :]).squeeze(-1)   # use the last time step


model = LSTMRegressor()
xb, yb = next(iter(train_loader))
print(xb.shape, yb.shape, model(xb).shape)
print("Parameters:", sum(p.numel() for p in model.parameters()))

torch.Size([128, 30, 28]) torch.Size([128]) torch.Size([128])
Parameters: 59457


In [2]:
y_val_np = d["y_val"]
y_test_np = d["y_test"]

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))

def nasa_score(y_true, y_pred):
    diff = y_pred - y_true
    return float(np.sum(np.where(diff < 0, np.exp(-diff / 13) - 1, np.exp(diff / 10) - 1)))

def predict(model, X):
    model.eval()
    with torch.no_grad():
        return model(X).numpy() * RUL_CAP

model = LSTMRegressor()          # fresh, untrained model
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

EPOCHS = 30
best_val, best_state = float("inf"), None

for epoch in range(1, EPOCHS + 1):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()
    val_rmse = rmse(y_val_np, predict(model, X_val))
    if val_rmse < best_val:
        best_val = val_rmse
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    print(f"epoch {epoch:2d}  val RMSE {val_rmse:6.2f}")

model.load_state_dict(best_state)
test_pred = predict(model, X_test)
print(f"\nBest val RMSE: {best_val:.1f}")
print(f"Test RMSE: {rmse(y_test_np, test_pred):.1f}   Test score: {nasa_score(y_test_np, test_pred):.0f}")

epoch  1  val RMSE  21.48
epoch  2  val RMSE  17.17
epoch  3  val RMSE  15.88
epoch  4  val RMSE  15.96
epoch  5  val RMSE  15.84
epoch  6  val RMSE  14.60
epoch  7  val RMSE  14.07
epoch  8  val RMSE  14.14
epoch  9  val RMSE  14.93
epoch 10  val RMSE  14.42
epoch 11  val RMSE  14.23
epoch 12  val RMSE  14.01
epoch 13  val RMSE  13.55
epoch 14  val RMSE  14.01
epoch 15  val RMSE  13.36
epoch 16  val RMSE  13.37
epoch 17  val RMSE  14.20
epoch 18  val RMSE  13.33
epoch 19  val RMSE  13.41
epoch 20  val RMSE  13.09
epoch 21  val RMSE  13.24
epoch 22  val RMSE  13.26
epoch 23  val RMSE  13.23
epoch 24  val RMSE  12.77
epoch 25  val RMSE  13.25
epoch 26  val RMSE  13.06
epoch 27  val RMSE  12.87
epoch 28  val RMSE  14.22
epoch 29  val RMSE  13.68
epoch 30  val RMSE  13.10

Best val RMSE: 12.8
Test RMSE: 14.8   Test score: 424


In [4]:
import json, os

os.makedirs("../results", exist_ok=True)
os.makedirs("../models", exist_ok=True)

lstm_results = {
    "lstm": {
        "val_rmse": best_val,
        "test_rmse": rmse(y_test_np, test_pred),
        "test_score": nasa_score(y_test_np, test_pred),
        "epochs": EPOCHS,
        "note": "single run, seed 42, best epoch chosen on validation",
    }
}
with open("../results/lstm_fd001.json", "w") as f:
    json.dump(lstm_results, f, indent=2)

torch.save(model.state_dict(), "../models/lstm_fd001.pt")
print(open("../results/lstm_fd001.json").read())

{
  "lstm": {
    "val_rmse": 12.76699161529541,
    "test_rmse": 14.826937675476074,
    "test_score": 424.04742431640625,
    "epochs": 30,
    "note": "single run, seed 42, best epoch chosen on validation"
  }
}


In [5]:
import time

def train_one(seed, epochs=30):
    torch.manual_seed(seed)
    np.random.seed(seed)
    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=128, shuffle=True)
    m = LSTMRegressor()
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()
    best_v, best_s = float("inf"), None
    for _ in range(epochs):
        m.train()
        for xb, yb in loader:
            opt.zero_grad()
            loss_fn(m(xb), yb).backward()
            opt.step()
        v = rmse(y_val_np, predict(m, X_val))
        if v < best_v:
            best_v = v
            best_s = {k: t.clone() for k, t in m.state_dict().items()}
    m.load_state_dict(best_s)
    p = predict(m, X_test)
    return best_v, rmse(y_test_np, p), nasa_score(y_test_np, p)

rows = []
for seed in [1, 2, 3, 4, 5]:
    t0 = time.time()
    rows.append(train_one(seed))
    print(f"seed {seed}: val {rows[-1][0]:.1f}  test {rows[-1][1]:.1f}  score {rows[-1][2]:.0f}  ({time.time() - t0:.0f}s)")

arr = np.array(rows)
print(f"\nMean +/- std over {len(rows)} seeds")
print(f"val RMSE   {arr[:, 0].mean():.1f} +/- {arr[:, 0].std():.1f}")
print(f"test RMSE  {arr[:, 1].mean():.1f} +/- {arr[:, 1].std():.1f}")
print(f"test score {arr[:, 2].mean():.0f} +/- {arr[:, 2].std():.0f}")

seed 1: val 13.0  test 14.5  score 389  (247s)
seed 2: val 13.0  test 14.8  score 412  (243s)
seed 3: val 12.9  test 14.4  score 356  (278s)
seed 4: val 13.1  test 14.4  score 341  (248s)
seed 5: val 13.2  test 14.9  score 433  (244s)

Mean +/- std over 5 seeds
val RMSE   13.0 +/- 0.1
test RMSE  14.6 +/- 0.2
test score 386 +/- 34
